In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.image import resize

In [ ]:
# Load the trained model
model = tf.keras.models.load_model("./second_genre_classification_model.h5")

In [ ]:
model.summary()

In [ ]:
classes = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

In [ ]:
# Single Audio File Testing

In [ ]:
# Load and Process the audio file
def load_and_preprocess_file(file_path, target_shape=(128,173)):
    data=[]
    audio_data, sample_rate = librosa.load(file_path, sr=None)
    chunk_duration =4
    overlap_duration = 2

    chunk_samples = chunk_duration * sample_rate
    overlap_samples = overlap_duration * sample_rate
   
    # Tính toán số lượng khung
    num_chunks = int(np.ceil((len(audio_data) - chunk_samples) / (chunk_samples - overlap_samples))) + 1

    for i in range(num_chunks):
        start = i * (chunk_samples - overlap_samples)
        end = start + chunk_samples
        chunk = audio_data[start:end]
        mel_spectrogram = librosa.feature.melspectrogram(y=chunk, sr=sample_rate,n_mels=128)
        mel_db=librosa.power_to_db(mel_spectrogram, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        if mel_db.shape[1] >= target_shape[1]:
                mel_db = mel_db[:, :target_shape[1]]
        else:
            pad_width = target_shape[1] - mel_db.shape[1]
            mel_db = np.pad(mel_db, pad_width=((0,0), (0,pad_width)), mode='constant')

        # append mel spectrogram to data list and corresponding label to labels list
        data.append(np.expand_dims(mel_db, axis=-1))
        
    return np.array(data)

In [ ]:
file_path = "../Data_Test/atlasaudio-disco-518074.mp3"
# file_path = "../Data/genres_original/hiphop/hiphop.00003.wav"

In [ ]:
# playing a sound
from IPython.display import Audio
y, sr = librosa.load(file_path, sr=None)
Audio(y, rate=sr)


In [ ]:
X_test = load_and_preprocess_file(file_path)

In [ ]:
X_test.shape

In [ ]:
def model_prediction(X_test):
    X_test_resized = tf.image.resize(X_test, [128, 173])
    y_pred = model.predict(X_test)
    predicted_categories=np.argmax(y_pred, axis=1)
    unique_elements, counts_elements = np.unique(predicted_categories, return_counts=True)
    max_count=np.max(counts_elements)
    max_element=unique_elements[counts_elements==max_count]
    return  max_element[0]

In [ ]:
c_index = model_prediction(X_test)

In [ ]:
c_index

In [ ]:
print("Model Prediction: Model predicted genre -->", classes[c_index])

TEST MÔ HÌNH TRAIN LẦN 3